In [3]:
from langchain_community.llms.ctransformers import CTransformers
from transformers import AutoTokenizer, AutoModel
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document
from langchain.docstore.in_memory import InMemoryDocstore  
import faiss
import torch
import numpy as np

llm = CTransformers(
    model="TheBloke/Llama-2-7B-Chat-GGUF",
    model_type="llama",
    base="llama2",
    local_files_only=True
)

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

def embed_texts(texts):
    inputs = tokenizer(texts, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
    
    embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)
    return embeddings

test_embedding = embed_texts(["test"])
embedding_dim = test_embedding.shape[1] if len(test_embedding.shape) > 1 else 384

index = faiss.IndexFlatIP(embedding_dim)

docstore = InMemoryDocstore()
index_to_docstore_id = {}

vectorstore = FAISS(
    embedding_function=embed_texts, 
    index=index, 
    docstore=docstore, 
    index_to_docstore_id=index_to_docstore_id
)

documents = [
    Document(page_content="OpenAI is a framework for developing AI applications."),
    Document(page_content="LangChain provides a standard interface for chains."),
    Document(page_content="LangChain helps you develop AI applications efficiently."),
]

texts = [doc.page_content for doc in documents]
embeddings = embed_texts(texts)

for i, embedding in enumerate(embeddings):
    index.add(np.array([embedding], dtype=np.float32))
    index_to_docstore_id[i] = documents[i]

def fetch_all_documents():
    print("Stored Documents in FAISS:")
    print(f"Number of vectors in FAISS index: {index.ntotal}")
    for i, doc_id in index_to_docstore_id.items():
        print(f"Index {i}: {doc_id.page_content}")

def simple_retriever(query):
    query_embedding = embed_texts([query])
    query_embedding /= np.linalg.norm(query_embedding, axis=1, keepdims=True)
    D, I = index.search(query_embedding, k=1)
    print("Search Results:", I, D)
    if len(I) == 0 or I[0][0] == -1:
        return "Not found"
    retrieved_index = I[0][0]
    retrieved_distance = D[0][0]
    print(f"Retrieved index: {retrieved_index}, Distance: {retrieved_distance}")
    if retrieved_distance < 0.5:
        return "Not found"
    retrieved_doc = index_to_docstore_id.get(retrieved_index, None)
    if retrieved_doc:
        return retrieved_doc.page_content
    return "Not found"

class SimpleRetrievalQA:
    def __init__(self, llm, retriever):
        self.llm = llm
        self.retriever = retriever
    def run(self, query):
        context = self.retriever(query)
        print(f"\nContext Retrieved: {context}\n")
        if context == "Not found":
            return "I'm sorry, I couldn't find relevant information in the available documents."
        response = self.llm(f"Use only the provided context to answer the question.\n\n"
                            f"Context: {context}\n\n"
                            f"Question: {query}\n\n"
                            f"Answer:")
        return response

fetch_all_documents()

qa_chain = SimpleRetrievalQA(llm=llm, retriever=simple_retriever)
question = "LangChain helps you?"
answer = qa_chain.run(question)
print("\nFinal Answer:", answer)


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


Stored Documents in FAISS:
Number of vectors in FAISS index: 3
Index 0: OpenAI is a framework for developing AI applications.
Index 1: LangChain provides a standard interface for chains.
Index 2: LangChain helps you develop AI applications efficiently.
Search Results: [[2]] [[0.59873056]]
Retrieved index: 2, Distance: 0.5987305641174316

Context Retrieved: LangChain helps you develop AI applications efficiently.


Final Answer:  Develop AI applications efficiently.
